In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys
from tqdm import tqdm

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT)) 

from roi_classifier.prepare_data import prepare_roi_data
from roi_classifier.annotate_data import annotate_rois
from roi_classifier.train_classifier import train_roi_classifier




In [2]:
DATASET_ROOT = Path(r"C:\Users\mzinn1\Desktop\Morgan 1-20-26")  # TODO: set this to your data path
assert DATASET_ROOT.exists(), f"Dataset root {DATASET_ROOT} does not exist."

ROI_DIR = PROJECT_ROOT / "data"
ROI_DIR.mkdir(parents=True, exist_ok=True)

ROI_DATA_PATH = ROI_DIR / "all_roi_features.npy"

MODEL_OUT_DIR = PROJECT_ROOT / "models"
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = PROJECT_ROOT / "config/classifier_config.yaml"

print(f"Extracting fluorescence data from {DATASET_ROOT.__str__()}")
print(f"Saving engineered data to {ROI_DATA_PATH.__str__()}")
print(f"Saving models to {MODEL_OUT_DIR.__str__()}")
print(f"Configuring classifier according to {CONFIG_PATH.__str__()}")

Extracting fluorescence data from C:\Users\mzinn1\Desktop\Morgan 1-20-26
Saving engineered data to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy
Saving models to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models
Configuring classifier according to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\classifier_config.yaml


In [3]:
update = True # Change to false 
backup = False # Change as you wish; controls whether or not a backup of the original engineered data is saved


roi_data = prepare_roi_data(
    dataset_root=DATASET_ROOT,
    input_file=ROI_DATA_PATH,
    output_file=ROI_DATA_PATH,
    update=update,
    backup=backup
)



  ROI Summary
  Total rois: 18652
  Good: 598 | Bad: 286 | Unlabeled: 17768
  Manual: 884 | Auto: 0
  Total spikes stored: 10775


Updated 18652 ROIs
  - Preserved 884 manual labels
  - Preserved 10775 spikes

Saved 18652 ROIs to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy


In [4]:
# Change these flags to control which ROIs are shown for annotation and how many
unlabeled_only = True 
labeled_only = False 
n_samples = 1000

assert not (unlabeled_only and labeled_only), "unlabeled_only and labeled_only cannot both be True — pick one or set both to False to show all ROIs."

annotate_rois(data_path=ROI_DATA_PATH,
              n_samples=n_samples,
              unlabeled_only=unlabeled_only,
              labeled_only=labeled_only)

Loaded 18652 ROIs from C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy
Found 17768 roi keys matching filter
Found 1000 unlabeled ROIs out of 17768 ROIs.
Session ended by user. Saving progress...
Saved to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy
Saved to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy

  ROI Annotation Summary
  Queued:    1000
  Seen:      0
  Labeled:   0
  Updated:   0
  Confirmed: 0
  Skipped:   0


  ROI Summary
  Total rois: 18652
  Good: 598 | Bad: 286 | Unlabeled: 17768
  Manual: 884 | Auto: 0
  Total spikes stored: 10775



{'level': 'roi',
 'queued': 1000,
 'total': 0,
 'labeled': 0,
 'updated': 0,
 'confirmed': 0,
 'skipped': 0}

In [5]:
name = "roi_classifier" # TODO Change this as needed for your own experimental/organizational needs 

results = train_roi_classifier(config_path=CONFIG_PATH, data_path=ROI_DATA_PATH, name="roi_classifier",
                     output_dir=MODEL_OUT_DIR, verbose=True, manual_only=True, overwrite=False)

Dataset Summary
--------------------------------------------------
Total labeled datapoints: 884
  Train: 707 | Test: 177

Label distribution:
              Bad (0)  Good (1)
  Train           236       471
  Test             50       127
  Total           286       598

Training on: Manual labels only

--------------------------------------------------
TUNED MODEL SUMMARY
--------------------------------------------------
Model:     RandomForestClassifier
Transform: sqrt
Features:  ['peak_density', 'range_trace', 'derivative_skew', 'ac_decay', 'var_of_var']

Hyperparameters:
  class_weight: None
  max_depth: 30
  min_samples_leaf: 1
  min_samples_split: 2
  n_estimators: 100

Metrics:
  CV Accuracy:   0.9731
  Test Accuracy: 0.9661
  ROC AUC:       0.9950
  F1:            0.9663
  Precision:     0.9668
  Recall:        0.9661

Confusion Matrix:
              Pred 0  Pred 1
  Actual 0    48      2      
  Actual 1    4       123    
--------------------------------------------------
Sa